In [1]:
import os

directory_path = '/kaggle/input/web-janak-qwen/other/default/1/webjanak-qwen-finetuned'

if os.path.exists(directory_path):
    print(f"Contents of {directory_path}:")
    for item in os.listdir(directory_path):
        print(item)
else:
    print(f"Directory not found: {directory_path}")

Contents of /kaggle/input/web-janak-qwen/other/default/1/webjanak-qwen-finetuned:
adapter_model.safetensors
merges.txt
training_args.bin
adapter_config.json
README.md
vocab.json
tokenizer_config.json
chat_template.jinja
checkpoint-200
special_tokens_map.json
added_tokens.json


In [2]:
!pip install -U bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 26.2 MB/s eta 0:00:0000:0100:01


In [4]:
# ============================================================
# WebJanak AI – (Base Model + LoRA)
# ============================================================

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

# -------------------------------
# CONFIG
# -------------------------------
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
LORA_PATH = "/kaggle/input/web-janak-qwen/other/default/1/webjanak-qwen-finetuned"

MAX_NEW_TOKENS = 1024
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------------
# LOAD TOKENIZER (BASE MODEL ONLY)
# -------------------------------
print("📝 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    use_fast=False
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# -------------------------------
# LOAD BASE MODEL (4-bit)
# -------------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("🤖 Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={"": 0},   # 👈 FORCE ALL MODULES TO GPU
    trust_remote_code=True
)

# -------------------------------
# ATTACH LORA
# -------------------------------
print("🔗 Attaching LoRA adapter...")
model = PeftModel.from_pretrained(
    model,
    LORA_PATH,
    device_map="auto"
)

model.eval()
print("✅ Model & LoRA loaded successfully!\n")

# ============================================================
# GENERATION FUNCTION
# ============================================================
def generate_ui(prompt, max_tokens=MAX_NEW_TOKENS):

    messages = [
        {
            "role": "system",
            "content": (
                "You are WebJanak AI, an expert React and HTML developer. "
                "Generate clean, modern, responsive UI code. "
                "Prefer semantic HTML and Tailwind-style layouts when possible."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Clean assistant output
    if "<|im_start|>assistant" in decoded:
        decoded = decoded.split("<|im_start|>assistant")[-1]

    return decoded.strip()

# ============================================================
# TEST PROMPTS
# ============================================================
test_prompts = [
    "Create a modern portfolio website with hero section",
    "Build a coffee shop landing page with menu",
    "Design a responsive admin dashboard with stats cards"
]

print("🧪 Testing WebJanak AI...\n")

for i, prompt in enumerate(test_prompts, 1):
    print(f"🔹 Test {i}/{len(test_prompts)}")
    print(f"🧠 Prompt: {prompt}")
    print("-" * 60)

    output = generate_ui(prompt)

    print(f"📏 Output Length: {len(output)} characters")
    print("\n📄 Preview (first 300 chars):\n")
    print(output[:3000])
    print("\n" + "=" * 60 + "\n")

print("🎉 All tests completed successfully!")


📝 Loading tokenizer...
🤖 Loading base model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🔗 Attaching LoRA adapter...
✅ Model & LoRA loaded successfully!

🧪 Testing WebJanak AI...

🔹 Test 1/3
🧠 Prompt: Create a modern portfolio website with hero section
------------------------------------------------------------
📏 Output Length: 3258 characters

📄 Preview (first 300 chars):

system
You are WebJanak AI, an expert React and HTML developer. Generate clean, modern, responsive UI code. Prefer semantic HTML and Tailwind-style layouts when possible.
user
Create a modern portfolio website with hero section
assistant
Certainly! Below is a modern portfolio-style webpage with a hero section, services section, work portfolio section, and client testimonials section. It uses semantic HTML and incorporates modern design principles:

```html
<!DOCTYPE html>
<html><head><meta charset="UTF-8"><title>Portfolio</title>
<style>
* { margin: 0; padding: 0; box-sizing: border-box; }
body { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Oxygen, Ubuntu, sans-serif; background: